# 02 — Aggregate Results

Reads all completed runs from `results/`, builds the pivot tables and
figures for the thesis.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scripts.aggregate import collect_runs

sns.set_style('whitegrid')
df = collect_runs()
print('Loaded', len(df), 'runs')
df.head()

## Model display name
Create a single column for plotting.

In [ ]:
def make_label(row):
    if row['model_type'] == 'classical':
        return row['model_name'].upper()
    return f"{row['encoding'][:3]}+{row['ansatz'][:5]}"

df['model_label'] = df.apply(make_label, axis=1)

## RQ1: Learning curves (Test F1 vs samples)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, scenario in zip(axes, ['S1', 'S3']):
    sub = df[df['scenario'] == scenario]
    if sub.empty:
        ax.set_title(f'{scenario} (no data)')
        continue
    grouped = sub.groupby(['model_label', 'train_samples_per_class'])['test_f1'].agg(['mean', 'std']).reset_index()
    for label, g in grouped.groupby('model_label'):
        ax.errorbar(g['train_samples_per_class'], g['mean'], yerr=g['std'],
                    marker='o', label=label, capsize=3)
    ax.set_xlabel('Samples per class')
    ax.set_ylabel('Test F1')
    ax.set_title(f'{scenario}')
    ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig('../figures/learning_curves.pdf', bbox_inches='tight')
plt.show()

## RQ2 + RQ3: Encoding and Ansatz comparison
Bar chart at 250 samples/class on S1.

In [ ]:
sub = df[(df.scenario == 'S1') & (df.train_samples_per_class == 250)]
agg = sub.groupby('model_label')[['test_f1', 'gap_f1', 'train_time_seconds']]\
         .agg(['mean', 'std']).round(4)
agg

## RQ-main: Cross-attack performance drop

In [ ]:
n = 250  # focus on one sample size for clarity
by_scenario = df[df.train_samples_per_class == n].groupby(['model_label', 'scenario'])['test_f1'].mean().unstack()
by_scenario['drop'] = by_scenario['S1'] - by_scenario['S3']
by_scenario['relative_drop'] = by_scenario['drop'] / by_scenario['S1']
by_scenario.sort_values('drop')

## Save thesis tables to CSV

In [ ]:
import os
os.makedirs('../thesis_tables', exist_ok=True)
agg.to_csv('../thesis_tables/table3_main_results.csv')
by_scenario.to_csv('../thesis_tables/table5_cross_attack.csv')
print('Saved')